In [ ]:
import time
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.chrome.options import Options

# =========================
# 1. 关键词
# =========================
keywords = [
    "西安 散步",
    "西安 公园",
    "西安 打卡",
    "西安 绿地",
    "西安 拍照",
    "西安 森林",
    "西安 草地",
    "西安 花园",
    "西安 游玩",
    "西安 郊游",
    "西安 风景",
    "西安 河",
    "西安 湖泊",
]

max_pages = 5   # 每个关键词翻页数（建议5~10）
output_file = "微博_西安绿地游憩_搜索结果.xlsx"

# =========================
# 工具函数：数字转换（关键）
# =========================
def parse_count(text):
    text = text.replace("赞", "").replace("转发", "").replace("评论", "").strip()
    if "万" in text:
        try:
            return float(text.replace("万", "")) * 10000
        except:
            return 0
    try:
        return int(text)
    except:
        return 0

# =========================
# 启动浏览器
# =========================
chrome_options = Options()
chrome_options.add_argument("--start-maximized")

driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=chrome_options)

all_data = []

try:
    driver.get("https://s.weibo.com/")
    print("👉 请先登录微博，然后回到这里")
    input("👉 登录完成后按回车继续...")

    for keyword in keywords:
        print(f"\n===== 开始抓取：{keyword} =====")

        search_url = f"https://s.weibo.com/weibo?q={keyword}&scope=ori&suball=1"
        driver.get(search_url)
        time.sleep(5)

        for page in range(max_pages):
            print(f"📄 第 {page+1} 页")

            # ✅ 渐进滚动（关键）
            for i in range(10):
                driver.execute_script(f"window.scrollTo(0, document.body.scrollHeight*{(i+1)/10});")
                time.sleep(1)

            cards = driver.find_elements(By.CSS_SELECTOR, "div.card-wrap")
            print(f"本页抓到 {len(cards)} 条")

            for card in cards:
                try:
                    text = ""
                    user_name = ""
                    post_time = ""
                    reposts = 0
                    comments = 0
                    likes = 0
                    post_url = ""

                    # 用户名
                    try:
                        user_elem = card.find_element(By.CSS_SELECTOR, "a.name")
                        user_name = user_elem.text.strip()
                    except:
                        pass

                    # 正文
                    try:
                        txt_elem = card.find_element(By.CSS_SELECTOR, "p.txt")
                        text = txt_elem.text.strip()
                    except:
                        pass

                    # 时间+链接
                    try:
                        from_elem = card.find_element(By.CSS_SELECTOR, "p.from a")
                        post_time = from_elem.text.strip()
                        post_url = from_elem.get_attribute("href")
                    except:
                        pass

                    # 转评赞（数值化）
                    action_texts = card.find_elements(By.CSS_SELECTOR, "div.card-act li")
                    if len(action_texts) >= 4:
                        reposts = parse_count(action_texts[1].text)
                        comments = parse_count(action_texts[2].text)
                        likes = parse_count(action_texts[3].text)

                    if text:
                        all_data.append({
                            "关键词": keyword,
                            "用户": user_name,
                            "发布时间": post_time,
                            "正文": text,
                            "转发": reposts,
                            "评论": comments,
                            "点赞": likes,
                            "链接": post_url
                        })

                except Exception as e:
                    print("❌ 单条失败:", e)

            # ✅ 翻页（关键）
            try:
                next_btn = driver.find_element(By.XPATH, "//a[contains(text(),'下一页')]")
                driver.execute_script("arguments[0].click();", next_btn)
                time.sleep(5)
            except:
                print("⚠️ 没有下一页了")
                break

    # =========================
    # 保存 + 统计
    # =========================
    df = pd.DataFrame(all_data)
    df.drop_duplicates(subset=["正文", "链接"], inplace=True)

    # 📊 统计
    summary = df.groupby("关键词").agg({
        "点赞": "sum",
        "评论": "sum",
        "转发": "sum"
    }).reset_index()

    # 写入Excel（双sheet）
    with pd.ExcelWriter(output_file) as writer:
        df.to_excel(writer, sheet_name="原始数据", index=False)
        summary.to_excel(writer, sheet_name="统计结果", index=False)

    print(f"\n✅ 抓取完成！文件已保存：{output_file}")

finally:
    driver.quit()
